# data_pipeline: Data pipelines for Pre-Training

## Learning Objectives
After studying and implementing this section, you should be able to:

1. Build a streaming data pipeline capable of tokenizing, chunking, shuffling, and batching terabytes of text without loading the entire dataset into memory.
2. Implement data quality filters such as deduplication, language detection, and content filtering, mirroring real-world pre-training pipelines.
3. Construct fixed-length training sequences, properly handling attention masks and document boundaries.
4. Measure and profile pipeline throughput to ensure the DataLoader feeds data fast enough to keep GPU compute fully utilized without starvation.


## What is the problem?
You have a tokenizer; now you need data.

This is not about a small dataset or a CSV file. To train a large language model, you need terabytes of text that has been cleaned, deduplicated, quality-filtered, and evaluated. Then, the text must be tokenized, divided into fixed-length sequences, and delivered to the model in random batches at sufficient speed. The pipeline's speed must be high enough that your eight-GPU cluster never has to wait for the next batch.

Many believe LLM training primarily depends on model architecture, but data plays a decisive role. Llama 3 was trained on 15.6 trillion tokens, GPT-3 on 300 billion tokens, and DeepSeek-V2 on 8.1 trillion tokens. The overall architecture of all three is more or less similar: multiple Transformer blocks stacked together, comprising attention and feed-forward layers. A significant portion of the difference in output quality among these models stems from the volume, composition, and quality of the training data.

DeepMind's Chinchilla paper elaborated on this. For any given compute budget, there is an optimal ratio between the number of model parameters and the number of training tokens. The research indicated that most models in 2022 were significantly undertrained, meaning they had too many parameters relative to the amount of data they were exposed to. For instance, a 70 billion parameter model trained on 1.4 trillion tokens, following the optimal Chinchilla ratio, outperformed Gopher, a 280 billion parameter model trained on only 300 billion tokens.

Therefore, your data pipeline determines whether the model truly learns language, knowledge, and useful patterns, or merely reproduces the noise present in the data.



In [14]:

import math
import re
import hashlib
import random
import time
from collections import Counter, defaultdict
from typing import List, Tuple, Set, Dict, Any, Generator, Optional

In [15]:
# basic clean text
def clean_text(text: str) -> str:
    """
    Cleans raw document text by stripping HTML tags, URLs, non-ASCII characters,
    and normalizing whitespace.

    Args:
        text (str): Input raw text document.

    Returns:
        str: Cleaned and normalized text string.
    """
    # TODO: Strip unwanted markup, non-ASCII noise, and normalize spaces/newlines.

    # if is NOT text-> return ""
    if not text:
        return ""

    # by regex remove tags
    text = re.sub(r"<[^>]+>", " ", text)

    # remove URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove ASCII
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # remove extera space in first and end
    text = re.sub(r"\s+", " ", text).strip()

    return text



Implementing this function in the pre-training pipeline simulates data quality filtering, screening out low-quality inputs such as promotional pages or spam content.

This function must evaluate three criteria:
- Word count
- Ratio of all-uppercase words
- Special character density

This function originates from the logic of well-known pipelines like RefinedWeb and Falcon, which check thresholds for length, uppercase ratio, and punctuation ratio for each document. These three criteria are aggregated to ensure each document passes through three gates.



In [16]:
def quality_filter(
    text: str, 
    min_words: int = 50, 
    max_ratio_caps: int = 0.3, 
    max_ratio_special: float = 0.1
) -> bool:
    """
    Filters out low-quality documents based on length, capitalization ratio, and special character density.

    Args:
        text (str): Cleaned document text.
        min_words (int): Minimum required word count.
        max_ratio_caps (float): Maximum allowed ratio of ALL-CAPS words.
        max_ratio_special (float): Maximum allowed ratio of non-alphanumeric special characters.

    Returns:
        bool: True if the document meets quality criteria, False otherwise.
    """
    # TODO: Check word count thresholds and measure capitalization and special-character ratios.
    
    # min vocab - Word count
    words = text.split()
    if len(words) < min_words:
        return False

    # Ratio of all-uppercase words
    caps_words = sum(
        1 for w in words
        if w.isalpha() and w == w.upper()
    )
    if caps_words / len(words) > max_ratio_caps:
        return False

    # Special character density
    n_special = sum(1 for ch in text if not ch.isalnum() and not ch.isspace())
    if n_special / len(text) > max_ratio_special:
        return False

    return True


This function serves as the foundation for the Deduplication algorithm (removing exact and near-duplicate documents) using MinHash LSH.

### Why Word-based Shingling (Word n-grams)?
- **What is a Shingle?** A continuous sequence of $k$ consecutive words (equivalent to a word-level $n$-gram with length $n = k$).
- **Why a Set?** In MinHash theory, a document is modeled as a "set" of shingles so that the Jaccard Similarity coefficient between two documents $A$ and $B$ can be computed:
  $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
- **Why Lowercase?** Variations in letter casing (such as "The" and "the") should not cause identical phrases to be treated as distinct; normalizing to `lower()` is essential.
- **Handling Short Texts:** If the document contains fewer than $k$ words, no shingle of length $k$ can be constructed; in this case, the function must return an empty set (`set()`).


### Implementation Steps
1. Convert the entire text to lowercase using `.lower()`.
2. Tokenize the text into a list of words using `.split()`.
3. Check the length condition: if `len(words) < k`, immediately return `set()`.
4. Iterate over the word list using a sliding window of length $k$, joining the tokens with a single space (`" ".join(...)`) to construct each shingle.
5. Collect and store all generated shingles in a unique `set`.


In [17]:
def get_shingles(text: str, k: int = 5) -> Set[str]:
    """
    Extracts k-shingles (word n-grams) from a text string.

    Args:
        text (str): Input document text.
        k (int): Size of word shingle (n-gram length).

    Returns:
        Set[str]: Unique set of k-word shingles.
    """
    # TODO: Lowercase, tokenize into words, and construct word n-gram shingles.
    if not text:
        return set()

    # Lower
    words = text.lower().split()

    # if k is graten than len(tex) -> cannot use
    if len(words) < k:
        return set()

    # Join
    shingles = {
        " ".join(words[i : i + k]) 
        for i in range(len(words) - k + 1)
    }

    return shingles


This function computes the MinHash signature (a fixed-length vector of size `num_hashes`) for a given set of shingles.

### MinHash Theory:
According to the MinHash theorem, the probability that the minimum hash value of two sets is equal under a random hash function $h_i$ is exactly equal to their Jaccard Similarity:

$$P(h_i(A) = h_i(B)) = J(A, B)$$

Therefore, given $m$ independent hash functions ($h_0, h_1, \dots, h_{m-1}$), the signature vector is constructed as follows:

$$\text{sig}[i] = \min_{s \in \text{shingles}} h_i(s)$$





In [18]:

# shingles from def get_shingles


def minhash_signature(shingles: Set[str], num_hashes: int = 128) -> List[int]:
    """
    Computes MinHash signature for a set of shingles using hash function permutations.

    Args:
        shingles (Set[str]): Set of unique word shingles.
        num_hashes (int): Length of the MinHash signature vector.

    Returns:
        List[int]: MinHash signature list of length (num_hashes,).
    """
    # TODO: Generate hash permutations for each seed and compute the minimum hash value per seed.

    if not shingles:
        return [0] * num_hashes

    signature = []

    for seed in range(num_hashes):
        min_val = float("inf")
        seed_bytes = str(seed).encode("utf-8")

        for s in shingles:
            
            # Combining a seed and a shingle to simulate an independent and deterministic hash function.
            h = hashlib.md5(seed_bytes + b"_" + s.encode("utf-8")).hexdigest()

            #Converting to an integer and optimizing processing speed
            
            val = int(h[:16], 16)

            if val < min_val:
                min_val = val

        signature.append(min_val)

    return signature


If we have $N$ documents, pairwise comparison of MinHash signatures requires $\frac{N(N-1)}{2}$ comparisons, resulting in a time complexity of $O(N^2)$.
The objective of LSH (Locality-Sensitive Hashing) is to reduce these comparisons: it maps documents with a high probability of similarity into a shared bucket, ensuring that only documents within the same bucket are compared.
A document's full signature consists of $M$ hashes (e.g., 128).

- If we enforce that "two documents must match across all 128 hashes," the condition becomes overly strict (retrieving only 100% identical copies).
- If we evaluate "each hash individually," the condition becomes overly relaxed, leading to a high number of dissimilar candidates (False Positives).

**LSH Mathematical Solution:** The signature is divided into $b$ bands, each containing $r$ numbers ($b \times r = M$).

The probability that two documents with Jaccard similarity $s$ become candidate pairs in at least one band is given by:

$$P(\text{Candidate}) = 1 - (1 - s^r)^b$$

This formula produces an S-curve that acts as a sharp threshold filter, identifying documents above the similarity threshold with high probability.

### Why convert to string and then hash with MD5?
```python
        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()
```

1. **Exact Matching Within a Band:** We want two documents to be assigned to the same bucket in a band if and only if all $r$ numbers in that band are strictly identical and in the exact same order.
2. **Fixed Size & Dictionary Key:** The sub-vector `chunk` is a Python list of integers. To look up this sub-vector in structures like `dict` or hash tables with $O(1)$ speed while minimizing memory overhead, we map it to a fixed-length hash string (32 Hex characters).


In [19]:
# signature from def minhash_signature

def lsh_buckets(signature: List[int], bands: int = 16) -> List[Tuple[int, str]]:
    """
    Divides MinHash signature into bands and maps each band to a bucket identifier using LSH.

    Args:
        signature (List[int]): MinHash signature vector of shape (num_hashes,).
        bands (int): Number of bands to divide the signature into.

    Returns:
        List[Tuple[int, str]]: List of (band_id, bucket_hash) tuples of length (bands,).
    """
    # TODO: Slice signature into bands, hash each band chunk, and map to bucket identifiers.
    num_hashes = len(signature)
    r = num_hashes // bands

    buckets = []
    for band_id in range(bands):

        chunk = signature[band_id * r : (band_id + 1) * r]

        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()

        buckets.append((band_id, bucket_hash))

    return buckets

**Storing `doc_shingles`:**   
We retain the shingles so that in the final step, rather than relying on an estimation, we compute the exact Jaccard similarity. LSH only identifies potential candidate pairs (Candidate Generation); however, because false positives can occur due to random collisions, final filtering must be performed using the exact formula:

$$\frac{|A \cap B|}{|A \cup B|}$$

**Using `defaultdict(list)`:**   
This provides the most efficient approach for grouping documents into LSH buckets. Each key serves as a "bucket address", and its corresponding value is a list of document indices mapped to that bucket.

**Preventing Redundant Computation (`already_compared`):**   
A pair of documents may land in the same bucket across multiple bands. By utilizing a `set` of sorted tuples, we guarantee that each document pair is evaluated for Jaccard similarity only once, preventing unnecessary CPU overhead.

**Removal Strategy (`to_remove`):**   
Instead of removing items dynamically on the fly (which disrupts index alignment), we collect redundant indices into a `set` and reconstruct the entire dataset in a single pass at the end using a list comprehension.

**Note:** In the line `jaccard = len(s1 & s2) / len(s1 | s2)`, a division-by-zero error occurs if both documents are empty. This is prevented using `if not s1 or not s2`.


In [20]:
def deduplicate(
    documents: List[str], 
    threshold: float = 0.8, 
    num_hashes: int = 128, 
    bands: int = 16
) -> Tuple[List[str], int]:
    """
    Finds and removes near-duplicate documents using MinHash LSH and Jaccard similarity evaluation.

    Args:
        documents (List[str]): List of clean document strings.
        threshold (float): Jaccard similarity threshold for considering two docs as duplicates.
        num_hashes (int): Total hash functions for MinHash computation.
        bands (int): Number of bands for LSH partitioning.

    Returns:
        Tuple[List[str], int]: Tuple containing (list of deduplicated documents, count of removed duplicates).
    """
    # TODO: Build MinHash signatures and map documents into LSH band buckets.
    # TODO: Collect candidate pairs from shared buckets, verify Jaccard similarity, and prune duplicates.
    if not documents:
        return [], 0

    # save new data
    # We keep the shingles so that in the final step
    # instead of an estimation, we compute the exact Jaccard similarity
    signatures = []
    doc_shingles = []

    for doc in documents:
        
        shingles = get_shingles(doc, k=5) # from def get_shingles(difalt k=5)
        doc_shingles.append(shingles)
        signatures.append(minhash_signature(shingles, num_hashes))

    # Mapping and make lsh_index
    lsh_index = defaultdict(list)
    for idx, sig in enumerate(signatures):
        buckets = lsh_buckets(sig, bands)
        for band_id, b_hash in buckets:
            lsh_index[(band_id, b_hash)].append(idx)

    # find 
    to_remove = set()
    already_compared = set()

    for candidates in lsh_index.values():
        if len(candidates) < 2:
            continue
            
        # Check all pairs in a bucket
        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                idx1, idx2 = candidates[i], candidates[j]
                
                # sort for dublicate
                pair = tuple(sorted((idx1, idx2)))
                if pair in already_compared or idx1 in to_remove or idx2 in to_remove:
                    continue
                
                already_compared.add(pair)
                # Verification
                s1, s2 = doc_shingles[idx1], doc_shingles[idx2]
                if not s1 or not s2: continue
                
                jaccard = len(s1 & s2) / len(s1 | s2)
                
                if jaccard >= threshold:
                    to_remove.add(idx2)

    # create list
    deduplicated_docs = [doc for i, doc in enumerate(documents) if i not in to_remove]
    
    return deduplicated_docs, len(to_remove)


The BPE (Byte Pair Encoding) algorithm is a data compression method for language models. The tasks performed in `train_bpe` serve the following purposes:

1. **Compression Efficiency**
Why frequency? We use `stats` to count which character/token pair occurs most frequently in the entire text. By merging the most frequent pair, we shorten the token sequence at each step.
**Goal:** The final model represents similar texts with fewer tokens, increasing speed and reducing memory usage.

2. **Hierarchical Tokenization**
Why perform replacement in the `new_tokens` loop? This is the most critical part. If we only registered merges, we would only learn 2-gram pairs. However, by performing replacement in `tokens`, new tokens are created. In the next step, the algorithm can combine these new tokens with others to form 3-gram, 4-gram, etc. tokens (e.g., a + b -> ab, then ab + c -> abc). This provides a hierarchical structure to the vocabulary.

3. **Managing OOV (Out-Of-Vocabulary)**
Why start from bytes? If we started from Unicode characters, our vocabulary would be excessively large, and we might encounter characters not seen during training. By starting from bytes (0-255), the initial vocabulary space is small and constant, and any string can be represented by these 256 bytes. Thus, OOV errors are effectively eliminated.

4. **Preventing Vocabulary Bloat**
Why the `if stats[best_pair] < 2` condition? If a pair appears only once in the text, merging it does not aid compression. Doing so only consumes vocabulary space without the created token likely being reused. This condition serves as a logical threshold to halt useless merges.


In [21]:
class SimpleTokenizer:
    """
    A minimal Byte Pair Encoding (BPE) tokenizer implementation operating over byte sequences.
    """
    def __init__(self, vocab_size: int = 256):
        """
        Initializes tokenizer vocabulary and internal mapping structures.

        Args:
            vocab_size (int): Target vocabulary capacity.
        """
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.next_id: int = 256
        self.eos_id: Optional[int] = None
        self.pad_id: int = 0

    def train_bpe(self, text: str, num_merges: int) -> None:
        """
        Iteratively finds the most frequent pair of tokens and merges them into a new token.

        Args:
            text (str): Training text corpus.
            num_merges (int): Number of BPE merge operations to execute.

        Returns:
            None
        """
        # TODO: Convert text to raw byte tokens and iteratively identify most frequent token pairs.
        # TODO: Register new merged tokens into vocabulary and replace occurrences in token stream.
        # TODO: Assign specialized End-of-Sequence (EOS) token ID.
        
        if not text:
            return

        # encode UTF-8
        tokens = list(text.encode("utf-8"))

        for _ in range(num_merges):
            if len(tokens) < 2:
                break

            # zip: fast to run
            # statitics
            stats: Dict[Tuple[int, int], int] = {}
            # add token 1 + token 2 nad zip to make a pair
            for pair in zip(tokens, tokens[1:]):
                stats[pair] = stats.get(pair, 0) + 1

            # if pair have 2 object
            if not stats:
                break

            # max repeite pari
            best_pair = max(stats, key=stats.get)

            # if have NOT 2 object
            if stats[best_pair] < 2:
                break

            # update merges and vocab and id
            idx = self.next_id
            self.merges[best_pair] = idx
            self.vocab[idx] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.next_id += 1

            # make new token
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and tokens[i] == best_pair[0]
                    and tokens[i + 1] == best_pair[1]
                ):
                    new_tokens.append(idx)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        self.eos_id = self.next_id
        self.vocab[self.eos_id] = b"<|endoftext|>"
        self.next_id += 1

    def encode(self, text: str) -> List[int]:
        """
        Encodes input text into a list of token IDs using pre-learned BPE merge rules.

        Args:
            text (str): Input text string.

        Returns:
            List[int]: Encoded list of token IDs.
        """
        # TODO: Convert input text to byte IDs and sequentially apply learned BPE merge rules.
        if not text:
            return []

        tokens = list(text.encode("utf-8"))

        # Applying merges in the exact order they were recorded during training
        for pair, new_id in self.merges.items():
            if len(tokens) < 2:
                break

            new_tokens = []
            i = 0
            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and tokens[i] == pair[0]
                    and tokens[i + 1] == pair[1]
                ):
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        return tokens

    def decode(self, ids: List[int]) -> str:
        """
        Decodes a list of token IDs back into a UTF-8 string.

        Args:
            ids (List[int]): List of token IDs.

        Returns:
            str: Decoded text representation.
        """
        # TODO: Reconstruct byte sequence from token IDs, skipping special tokens, and decode to UTF-8 text.
        # O(n**2) --> O(1)
        byte_sequence = bytearray()
        
        for id_ in ids:
            # Control tokens like `eos_id` are not part of the original text
            # therefore, they must be removed during text reconstruction to avoid introducing noise.
            if self.eos_id is not None and id_ == self.eos_id:
                continue
            
            # Retrieving the bytes corresponding to the ID from the vocabulary
            if id_ in self.vocab:
                byte_sequence.extend(self.vocab[id_])
        
        return byte_sequence.decode("utf-8", errors="replace")

    def vocab_size(self) -> int:
        """
        Returns the total vocabulary size including base bytes and merges.

        Returns:
            int: Total vocabulary size.
        """
        # TODO: Return total number of active entries in vocabulary.
        return len(self.vocab)


**Flat Stream**

In language models such as GPT (Causal LM), training input is typically a continuous, long sequence of tokens that is later chunked into fixed-length windows matching the context size (`context_length` or `seq_len`). This prevents memory overhead caused by extensive padding across separate documents.

**The Role of the `eos_id` Delimiter:**

Inserting the `tokenizer.eos_id` between documents is crucial so that the model learns during next-token prediction that documents are independent, and that the end of one document has no contextual or causal relationship with the start of the next.

**Using `.extend()`:**

Instead of repeatedly concatenating lists with `+` (which creates a complete copy of the list each time, running in $O(N^2)$ time), the `.extend()` method appends the new document's tokens in amortized $O(K)$ time.


In [22]:
def tokenize_corpus(documents: List[str], tokenizer: SimpleTokenizer) -> List[int]:
    """
    Tokenizes a list of documents into a single flat stream of token IDs separated by EOS markers.

    Args:
        documents (List[str]): List of document strings.
        tokenizer (SimpleTokenizer): Trained BPE tokenizer instance.

    Returns:
        List[int]: Flat list containing combined token sequence.
    """
    # TODO: Tokenize individual documents, append EOS markers, and flatten into a continuous sequence.
    flat_tokens: List[int] = []
    
    for doc in documents:
        doc_tokens = tokenizer.encode(doc)
        flat_tokens.extend(doc_tokens)
        
        if tokenizer.eos_id is not None:
            flat_tokens.append(tokenizer.eos_id)
            
    return flat_tokens


**Fixed-length Chunking (`seq_length`)**

Due to the matrix nature of attention computations and the necessity for batching on GPUs, transformer models require inputs with fixed tensor dimensions (`batch_size`, `seq_length`).

**Trailing Chunk Management:**

If the total token length is not an exact multiple of `seq_length`, the final chunk will have a shorter length. By appending a `pad_id` (e.g., a default value of 0), it is padded to the specified `seq_length`.

**Role of the Attention Mask:**

A value of 1 is assigned to actual tokens so the model attends to them (Self-Attention). A value of 0 is assigned to padding tokens so they are masked to $-\infty$ in the attention layer, ensuring they do not impose gradients or random biases on the model.


In [23]:
def pack_sequences(
    token_ids: List[int], 
    seq_length: int, 
    pad_id: int = 0
) -> Tuple[List[List[int]], List[List[int]]]:
    """
    Packs continuous token stream into fixed-length sequence chunks with corresponding attention masks.

    Args:
        token_ids (List[int]): Continuous flat list of token IDs.
        seq_length (int): Target length per sequence block.
        pad_id (int): Token ID used for padding incomplete trailing sequences.

    Returns:
        Tuple[List[List[int]], List[List[int]]]: Tuple of (padded_sequences, attention_masks)
            where each element has dynamic batch outer dimension and sequence shape (seq_length,).
    """
    # TODO: Chunk continuous token IDs into uniform blocks of fixed sequence length.
    # TODO: Apply padding to trailing sequence blocks and generate binary attention mask indicators.
    if not token_ids or seq_length <= 0:
        return [], []

    padded_sequences: List[List[int]] = []
    attention_masks: List[List[int]] = []

    # Iterating through tokens with fixed step sizes of `seq_length`
    for i in range(0, len(token_ids), seq_length):
        chunk = token_ids[i : i + seq_length]
        chunk_len = len(chunk)

        if chunk_len == seq_length:
            # Full block without padding
            padded_sequences.append(chunk)
            attention_masks.append([1] * seq_length)
        else:
            # padding
            pad_amount = seq_length - chunk_len
            padded_chunk = chunk + [pad_id] * pad_amount
            mask = [1] * chunk_len + [0] * pad_amount

            padded_sequences.append(padded_chunk)
            attention_masks.append(mask)

    return padded_sequences, attention_masks

Training Loop Management: In PyTorch and any other machine learning framework, the training loop needs to know exactly how many steps to take in each epoch to cover all data.   
Monitoring Tools and Schedulers: Libraries like `tqdm` (progress bar) or Learning Rate Schedulers directly depend on the return value of the `__len__` method to compute the total number of iterations across the entire training process.   

`len(self.sequences)` determines the total number of packed sequence blocks.   
If the total number of samples is, for example, 1000 and the `batch_size` is 32, a simple division of 1000 by 32 yields 31.25.
Since a fraction of a batch (0.25) cannot exist, the final batch will contain the remaining samples (19 samples). The `math.ceil` function rounds this fraction up to the next integer (32), ensuring that no data is left behind at the end of the epoch.

---

Iteration Protocol: Implementing this method turns the class into an iterable structure, allowing it to be used directly inside the training loop (`for batch in dataloader:`).
Shuffling: Randomizing the order of samples at the beginning of each epoch prevents the model from memorizing fixed patterns in sequence order, improving gradient dynamics and generalization.

Index List Generation: `indices` contains the integer indices of the samples ranging from `0` to `len(sequences) - 1`.
Conditional Shuffling: If `shuffle=True`, the index order is randomized using `random.shuffle` (maintaining parallel alignment between `sequences` and `attention_masks`).
Batch Striding and Extraction: The loop iterates over indices with a step size of `batch_size`, extracts the corresponding slices, and yields them using a generator to optimize memory efficiency.



In [24]:
class PreTrainingDataLoader:
    """
    Data loader providing mini-batch iteration and shuffling over packed token sequences.
    """
    def __init__(
        self, 
        sequences: List[List[int]], 
        attention_masks: List[List[int]], 
        batch_size: int, 
        shuffle: bool = True
    ):
        """
        Initializes dataloader properties.

        Args:
            sequences (List[List[int]]): Packed input sequences of shape (num_samples, seq_length).
            attention_masks (List[List[int]]): Attention masks of shape (num_samples, seq_length).
            batch_size (int): Size of individual mini-batches.
            shuffle (bool): Whether to shuffle sample indices prior to batching.
        """
        self.sequences = sequences
        self.attention_masks = attention_masks
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self) -> int:
        """
        Calculates total number of batches per epoch.

        Returns:
            int: Number of available mini-batches.
        """
        # TODO: Compute total batch count accounting for ceiling division of sequence length.
        return math.ceil(len(self.sequences) / self.batch_size)

    def __iter__(self) -> Generator[Tuple[List[List[int]], List[List[int]]], None, None]:
        """
        Iterates over the dataset and yields mini-batches of sequences and attention masks.

        Returns:
            Generator[Tuple[List[List[int]], List[List[int]]], None, None]: Batch generator yielding 
            (batch_sequences, batch_attention_masks) where each tensor component has shape (batch_size, seq_length).
        """
        # TODO: Shuffle document indices conditionally, iterate in mini-batch strides, and yield batch slices.
        indices = list(range(len(self.sequences)))

        if self.shuffle:
            random.shuffle(indices)

        for i in range(0, len(indices), self.batch_size):
            batch_indices = indices[i:i + self.batch_size]

            batch_sequences = [self.sequences[idx] for idx in batch_indices]
            batch_attention_masks = [self.attention_masks[idx] for idx in batch_indices]

            yield batch_sequences, batch_attention_masks

In [25]:
def compute_statistics(
    documents: List[str], 
    token_ids: List[int], 
    sequences: List[List[int]], 
    tokenizer_vocab_size: int
) -> Dict[str, Any]:
    """
    Computes overall corpus, vocabulary, compression, and sequence padding efficiency metrics.

    Args:
        documents (List[str]): Corpus of cleaned documents.
        token_ids (List[int]): Continuous flat list of token IDs.
        sequences (List[List[int]]): Packed token sequences.
        tokenizer_vocab_size (int): Size of the tokenizer vocabulary.

    Returns:
        Dict[str, Any]: Dictionary containing pre-training dataset analytical statistics.
    """
    # TODO: Measure total corpus dimensions, compute character-to-token compression, and track document lengths.
    # TODO: Calculate top token frequencies, sequence padding efficiency ratios, and vocabulary utilization.
    num_documents = len(documents)
    total_characters = sum(len(doc) for doc in documents)
    total_tokens = len(token_ids)
    
    # len doc per words
    total_words = sum(len(doc.split()) for doc in documents)
    avg_doc_length_words = total_words / num_documents if num_documents > 0 else 0.0

    # rate comperes
    compression_ratio = total_characters / total_tokens if total_tokens > 0 else 0.0

    # Token Frequency and Vocabulary Efficiency
    token_counts = Counter(token_ids)
    unique_tokens_used = len(token_counts)
    vocab_utilization = unique_tokens_used / tokenizer_vocab_size if tokenizer_vocab_size > 0 else 0.0

    # Padding Efficiency in Packed Sequences
    num_sequences = len(sequences)
    total_packed_tokens = sum(len(seq) for seq in sequences)
    sequence_utilization = total_tokens / total_packed_tokens if total_packed_tokens > 0 else 0.0

    return {
        "total_documents": num_documents,
        "total_characters": total_characters,
        "total_tokens": total_tokens,
        "unique_tokens": unique_tokens_used,
        "vocab_utilization": vocab_utilization,
        "compression_ratio": compression_ratio,
        "avg_doc_length_words": avg_doc_length_words,
        "num_sequences": num_sequences,
        "sequence_utilization": sequence_utilization,
    }

In [26]:
# [KEEP_IMPLEMENTATION]
def generate_sample_corpus() -> List[str]:
    """
    Generates a pre-defined synthetic corpus containing standard, duplicate, HTML, and spam documents.

    Returns:
        List[str]: List of raw document strings.
    """
    base_docs = [
        "Machine learning is a subset of artificial intelligence that provides systems the ability "
        "to automatically learn and improve from experience without being explicitly programmed. "
        "Machine learning focuses on the development of computer programs that can access data and "
        "use it to learn for themselves. The process of learning begins with observations or data, "
        "such as examples, direct experience, or instruction, in order to look for patterns in data "
        "and make better decisions in the future based on the examples that we provide.",

        "Deep learning is part of a broader family of machine learning methods based on artificial "
        "neural networks with representation learning. Learning can be supervised, semi-supervised "
        "or unsupervised. Deep learning architectures such as deep neural networks, recurrent neural "
        "networks, convolutional neural networks and transformers have been applied to fields "
        "including natural language processing, speech recognition, computer vision, and many other tasks.",

        "Natural language processing is a subfield of linguistics, computer science, and artificial "
        "intelligence concerned with the interactions between computers and human language, in "
        "particular how to program computers to process and analyze large amounts of natural language "
        "data. The result is a computer capable of understanding the contents of documents, including "
        "the contextual nuances of the language within them.",

        "Transformers are a type of neural network architecture that has become the dominant approach "
        "for natural language processing tasks. The key innovation is the self-attention mechanism, "
        "which allows the model to weigh the importance of different parts of the input when producing "
        "each part of the output. This enables transformers to capture long-range dependencies in text "
        "much more effectively than previous recurrent approaches.",

        "The attention mechanism in neural networks allows the model to focus on relevant parts of "
        "the input sequence when generating each element of the output. In the transformer architecture, "
        "multi-head attention computes attention in parallel across multiple representation subspaces, "
        "enabling the model to jointly attend to information from different representation subspaces "
        "at different positions in the sequence.",

        "Reinforcement learning is an area of machine learning concerned with how intelligent agents "
        "ought to take actions in an environment in order to maximize the notion of cumulative reward. "
        "Reinforcement learning is one of three basic machine learning paradigms, alongside supervised "
        "learning and unsupervised learning. It differs from supervised learning in that correct input "
        "and output pairs need not be presented.",

        "Computer vision is an interdisciplinary scientific field that deals with how computers can "
        "gain high-level understanding from digital images or videos. From the perspective of "
        "engineering, it seeks to understand and automate tasks that the human visual system can do. "
        "Computer vision tasks include methods for acquiring, processing, analyzing and understanding "
        "digital images, and extraction of high-dimensional data from the real world.",

        "Convolutional neural networks are a class of deep learning architecture commonly applied to "
        "analyze visual imagery. They use a variation of multilayer perceptrons designed to require "
        "minimal preprocessing. They are also known as shift invariant or space invariant artificial "
        "neural networks based on their shared-weights architecture and translation invariance "
        "characteristics. Convolutional networks were inspired by biological processes.",

        "Generative adversarial networks consist of two neural networks that contest with each other "
        "in the form of a zero-sum game, where one agent gain is another agent loss. Given a training "
        "set, this technique learns to generate new data with the same statistics as the training set. "
        "For example, a generative adversarial network trained on photographs can generate new "
        "photographs that look authentic to human observers.",

        "Transfer learning is a machine learning method where a model developed for a task is reused "
        "as the starting point for a model on a second task. It is a popular approach in deep learning "
        "where pre-trained models are used as the starting point on computer vision and natural language "
        "processing tasks given the vast compute and time resources required to develop neural network "
        "models on these problems and the large improvements they provide.",
    ]

    near_dup_1 = (
        "Machine learning is a subset of artificial intelligence that provides systems the ability "
        "to automatically learn and improve from experience. Machine learning focuses on developing "
        "computer programs that can access data and use it to learn for themselves. The learning "
        "process begins with observations or data, such as examples or direct experience, in order "
        "to look for patterns and make better decisions based on the examples provided."
    )

    near_dup_2 = (
        "Deep learning is part of a broader family of machine learning methods based on artificial "
        "neural networks with representation learning. Learning can be supervised, semi-supervised "
        "or unsupervised. Deep learning architectures such as deep neural networks, recurrent neural "
        "networks, convolutional neural networks and transformers have been applied to fields "
        "including natural language processing, speech recognition, computer vision, and many other tasks."
    )

    short_doc = "This is too short to be useful."

    html_doc = (
        "<html><body><h1>Title</h1><p>Machine learning is transforming how we build software. "
        "Deep neural networks can learn complex patterns from data. The transformer architecture "
        "has become the dominant approach for language tasks. Self-attention allows models to capture "
        "long-range dependencies. Pre-training on large corpora produces strong foundation models. "
        "Fine-tuning adapts these models to specific tasks with minimal additional data.</p></body></html>"
    )

    spam_doc = "BUY NOW CLICK HERE FREE MONEY GUARANTEED RESULTS " * 20

    docs = base_docs + [near_dup_1, near_dup_2, short_doc, html_doc, spam_doc]
    return docs


# [KEEP_IMPLEMENTATION]
def run_pipeline():
    """
    Executes end-to-end data processing pipeline for pre-training dataset preparation.
    """
    print("=" * 60)
    print("Data Pipeline for Pre-Training")
    print("=" * 60)

    raw_docs = generate_sample_corpus()
    print(f"\nRaw documents: {len(raw_docs)}")

    print("\n--- Stage 1: Cleaning ---")
    cleaned_docs = [clean_text(doc) for doc in raw_docs]
    print(f"After HTML stripping: {len(cleaned_docs)} documents")

    print("\n--- Stage 2: Quality Filtering ---")
    filtered_docs = [doc for doc in cleaned_docs if quality_filter(doc)]
    removed_quality = len(cleaned_docs) - len(filtered_docs)
    print(f"Removed {removed_quality} low-quality documents")
    print(f"Remaining: {len(filtered_docs)} documents")

    print("\n--- Stage 3: Deduplication (MinHash + LSH) ---")
    start = time.time()
    deduped_docs, num_removed = deduplicate(filtered_docs, threshold=0.8)
    dedup_time = time.time() - start
    print(f"Removed {num_removed} near-duplicates in {dedup_time:.2f}s")
    print(f"Remaining: {len(deduped_docs)} documents")

    print("\n--- Stage 4: Tokenization ---")
    all_text = " ".join(deduped_docs)
    tokenizer = SimpleTokenizer()
    start = time.time()
    tokenizer.train_bpe(all_text, num_merges=100)
    train_time = time.time() - start
    print(f"Trained tokenizer with {tokenizer.vocab_size()} tokens in {train_time:.2f}s")

    start = time.time()
    token_ids = tokenize_corpus(deduped_docs, tokenizer)
    tok_time = time.time() - start
    print(f"Tokenized {len(token_ids):,} tokens in {tok_time:.2f}s ({len(token_ids)/max(tok_time, 0.001):,.0f} tokens/sec)")

    print("\n--- Stage 5: Sequence Packing ---")
    seq_length = 128
    sequences, masks = pack_sequences(token_ids, seq_length, pad_id=0)
    print(f"Packed into {len(sequences)} sequences of length {seq_length}")

    print("\n--- Stage 6: DataLoader ---")
    batch_size = 4
    loader = PreTrainingDataLoader(sequences, masks, batch_size)
    print(f"DataLoader: {len(loader)} batches of size {batch_size}")

    batch_count = 0
    total_tokens_served = 0
    for batch_seqs, batch_masks in loader:
        batch_count += 1
        total_tokens_served += sum(sum(m) for m in batch_masks)
        if batch_count <= 2:
            print(f"\n  Batch {batch_count}:")
            print(f"    Sequences: {len(batch_seqs)}")
            print(f"    First seq (first 20 tokens): {batch_seqs[0][:20]}...")
            print(f"    First mask (first 20): {batch_masks[0][:20]}...")
    print(f"\n  Total batches served: {batch_count}")
    print(f"  Total non-padding tokens served: {total_tokens_served:,}")

    print("\n--- Dataset Statistics ---")
    stats = compute_statistics(deduped_docs, token_ids, sequences, tokenizer.vocab_size())
    print(f"  Documents:           {stats['total_documents']}")
    print(f"  Total characters:    {stats['total_characters']:,}")
    print(f"  Total tokens:        {stats['total_tokens']:,}")
    print(f"  Unique tokens:       {stats['unique_tokens']}")
    print(f"  Vocab utilization:  {stats['vocab_utilization']:.1%}")
    print(f"  Compression ratio:  {stats['compression_ratio']:.2f} chars/token")
    print(f"  Avg doc length:     {stats['avg_doc_length_words']:.0f} words")
    print(f"  Num sequences:      {stats['num_sequences']}")
    print(f"  Seq utilization:    {stats['sequence_utilization']:.1%}")

    print("\n--- Pipeline Summary ---")
    print(f"  Raw documents:       {len(raw_docs)}")
    print(f"  After cleaning:      {len(cleaned_docs)}")
    print(f"  After quality filter: {len(filtered_docs)} (-{removed_quality})")
    print(f"  After dedup:         {len(deduped_docs)} (-{num_removed})")
    print(f"  Final tokens:        {len(token_ids):,}")
    print(f"  Training sequences:  {len(sequences)}")
    print(f"  Training batches:    {len(loader)}")


# ===== UNIT TESTS =====

def test_clean_text():
    sample = "<html><body>Hello    World! http://example.com</body></html>"
    cleaned = clean_text(sample)
    assert isinstance(cleaned, str), "Output must be a string."
    assert "<html>" not in cleaned, "HTML tags were not removed."
    assert "http" not in cleaned, "URLs were not stripped."
    assert "  " not in cleaned, "Multiple spaces were not collapsed."
    
    # Edge case: Empty input string
    assert clean_text("") == "", "Edge case failed: Empty string must return an empty string."


def test_quality_filter():
    good_text = " ".join(["word"] * 60)
    bad_text_short = "short text"
    bad_text_caps = " ".join(["WORD"] * 60)
    
    assert quality_filter(good_text) is True, "Valid document failed quality filter."
    assert quality_filter(bad_text_short) is False, "Short document incorrectly passed quality filter."
    assert quality_filter(bad_text_caps) is False, "Overly-capitalized document passed quality filter."
    
    # Edge case: Boundary word count
    exact_words = " ".join(["word"] * 50)
    assert quality_filter(exact_words, min_words=50) is True, "Edge case failed: Exact min_words boundary missed."


def test_get_shingles():
    text = "the quick brown fox jumps over the lazy dog"
    shingles = get_shingles(text, k=3)
    assert isinstance(shingles, set), "Output should be a set."
    assert len(shingles) == 7, "Shingle extraction length mismatch."
    
    # Edge case: Text shorter than k
    short_text = "quick brown"
    assert get_shingles(short_text, k=5) == set(), "Edge case failed: Text shorter than k must return an empty set."


def test_minhash_signature():
    shingles = {"quick brown fox", "brown fox jumps"}
    sig = minhash_signature(shingles, num_hashes=32)
    assert isinstance(sig, list), "MinHash signature must be a list."
    assert len(sig) == 32, "Signature length must match requested num_hashes."
    
    # Edge case: Empty shingle set
    empty_sig = minhash_signature(set(), num_hashes=16)
    assert len(empty_sig) == 16, "Edge case failed: MinHash length mismatched for empty shingles."
    assert all(val == 0 for val in empty_sig), "Edge case failed: Empty shingle signature must default to 0s."


def test_lsh_buckets():
    random.seed(42)
    sig = [random.randint(0, 1000) for _ in range(128)]
    buckets = lsh_buckets(sig, bands=16)
    assert isinstance(buckets, list), "Output must be a list."
    assert len(buckets) == 16, "Number of generated buckets must match bands count."
    assert len(buckets[0]) == 2, "Each bucket element must be a (band_id, bucket_hash) tuple."


def test_deduplicate():
    docs = [
        "Natural language processing is a subfield of linguistics and computer science.",
        "Natural language processing is a subfield of linguistics and computer science.",
        "Computer vision allows computers to derive meaningful information from digital images."
    ]
    deduped, removed = deduplicate(docs, threshold=0.8)
    assert isinstance(deduped, list), "Deduplicated docs must be returned as a list."
    assert len(deduped) == 2, "Duplicate document was not removed."
    assert removed == 1, "Removed duplicate count is incorrect."
    
    # Edge case: Single document input
    single_doc = ["Single document text."]
    d_out, r_out = deduplicate(single_doc)
    assert len(d_out) == 1 and r_out == 0, "Edge case failed: Single document should yield no duplicates."


def test_simple_tokenizer():
    tok = SimpleTokenizer()
    text = "hello world hello world"
    tok.train_bpe(text, num_merges=2)
    
    encoded = tok.encode("hello world")
    assert isinstance(encoded, list), "Encoded tokens must be a list of IDs."
    assert len(encoded) > 0, "Encoded sequence should not be empty."
    
    decoded = tok.decode(encoded)
    assert isinstance(decoded, str), "Decoded output must be a string."
    assert decoded == "hello world", "Decoded text does not match original."
    
    # Edge case: Out of vocabulary / raw unseen bytes
    unseen = tok.encode("xyz123")
    assert tok.decode(unseen) == "xyz123", "Edge case failed: Unseen characters were not decoded back accurately."


def test_tokenize_corpus():
    tok = SimpleTokenizer()
    tok.train_bpe("test document", num_merges=1)
    docs = ["test", "document"]
    tokens = tokenize_corpus(docs, tok)
    
    assert isinstance(tokens, list), "Corpus tokens must be a list."
    assert tokens.count(tok.eos_id) == 2, "Corpus token sequence must include one EOS token per document."


def test_pack_sequences():
    tokens = list(range(10))
    seqs, masks = pack_sequences(tokens, seq_length=4, pad_id=0)
    
    assert len(seqs) == 3, "Packed sequence count mismatch."
    assert len(seqs[0]) == 4, "Sequence block length mismatch."
    assert seqs[2] == [8, 9, 0, 0], "Padding incorrect on last sequence block."
    assert masks[2] == [1, 1, 0, 0], "Attention mask incorrect on padded sequence block."
    
    # Edge case: Empty token input
    e_seqs, e_masks = pack_sequences([], seq_length=4)
    assert len(e_seqs) == 0 and len(e_masks) == 0, "Edge case failed: Empty tokens should return empty lists."


def test_pre_training_data_loader():
    seqs = [[1, 2, 3, 4] for _ in range(10)]
    masks = [[1, 1, 1, 1] for _ in range(10)]
    loader = PreTrainingDataLoader(seqs, masks, batch_size=4, shuffle=False)
    
    assert len(loader) == 3, "DataLoader batch count mismatch."
    
    batches = list(loader)
    first_b_seqs, first_b_masks = batches[0]
    assert len(first_b_seqs) == 4, "Batch size mismatch."
    assert len(first_b_masks) == 4, "Attention mask batch size mismatch."
    
    # Edge case: Batch size greater than dataset size
    loader_large = PreTrainingDataLoader(seqs[:2], masks[:2], batch_size=8)
    large_batches = list(loader_large)
    assert len(large_batches) == 1, "Edge case failed: Oversized batch should return exactly 1 mini-batch."
    assert len(large_batches[0][0]) == 2, "Edge case failed: Single batch size should match total available samples."


def test_compute_statistics():
    docs = ["one two three", "four five"]
    token_ids = [1, 2, 3, 4, 5]
    sequences = [[1, 2, 3], [4, 5, 0]]
    
    stats = compute_statistics(docs, token_ids, sequences, tokenizer_vocab_size=256)
    assert isinstance(stats, dict), "Statistics output must be a dictionary."
    assert stats["total_documents"] == 2, "Stat mismatch: Total documents."
    assert stats["total_tokens"] == 5, "Stat mismatch: Total tokens."
    assert "compression_ratio" in stats, "Missing metric: compression_ratio."
    assert 0.0 <= stats["sequence_utilization"] <= 1.0, "Sequence utilization ratio out of valid bounds [0, 1]."


if __name__ == "__main__":
    run_pipeline()

Data Pipeline for Pre-Training

Raw documents: 15

--- Stage 1: Cleaning ---
After HTML stripping: 15 documents

--- Stage 2: Quality Filtering ---
Removed 2 low-quality documents
Remaining: 13 documents

--- Stage 3: Deduplication (MinHash + LSH) ---
Removed 1 near-duplicates in 0.07s
Remaining: 12 documents

--- Stage 4: Tokenization ---
Trained tokenizer with 357 tokens in 0.06s
Tokenized 2,661 tokens in 0.03s (79,814 tokens/sec)

--- Stage 5: Sequence Packing ---
Packed into 21 sequences of length 128

--- Stage 6: DataLoader ---
DataLoader: 6 batches of size 4

  Batch 1:
    Sequences: 4
    First seq (first 20 tokens): [32, 297, 308, 112, 262, 265, 290, 293, 98, 273, 97, 100, 296, 102, 97, 109, 105, 108, 309, 290]...
    First mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...

  Batch 2:
    Sequences: 4
    First seq (first 20 tokens): [258, 282, 298, 283, 100, 46, 356, 84, 105, 116, 108, 256, 77, 348, 308, 312, 261, 115, 301, 109]...
    First ma